In [1]:
import sys
print(sys.executable)

/Users/derekmartinez/Documents/Farber Lab/Donor Integrative Study/Donor Integrative Analysis/.venv-scrna/bin/python


In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
from pathlib import Path
import sys
import mygene
import harmonypy as hm

NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != "scRNA-seq-scripts":
    NOTEBOOK_DIR = NOTEBOOK_DIR / "scRNA-seq-scripts"
PROJECT_ROOT = NOTEBOOK_DIR.parent
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(NOTEBOOK_DIR))

from src.scrna.abundance_analysis import (
    analyze_donor_cell_type_abundance,
    plot_credible_abundance_findings,
    plot_credible_effect_sizes,
)
from src.scrna.cmv_analysis import (
    CMV_PALETTE,
    ETHNICITY_PALETTE,
    fit_cmv_adjusted_models,
    plot_cmv_sensitivity,
    plot_cmv_stratified_proportions,
    prepare_cmv_data,
)


In [3]:
adata = sc.read_h5ad("52.h5ad")

if adata.raw is None:
    raise ValueError("The input H5AD must retain raw counts in adata.raw.X.")

adata.layers["counts"] = adata.raw.X

In [4]:
adata.var['ensembl_id'] = adata.var_names.to_list()

mg = mygene.MyGeneInfo()
results = mg.querymany(
    adata.var_names.tolist(),
    scopes='ensembl.gene',
    fields='symbol',
    species='human',
    as_dataframe=True
)

results_df = results.reset_index()
symbol_map = dict(zip(results_df['query'], results_df['symbol']))

new_names = [
    str(symbol_map.get(gene_id)) if pd.notna(
        symbol_map.get(gene_id)) else gene_id
    for gene_id in adata.var_names
]

adata.var_names = new_names
adata.var_names_make_unique()

29 input query terms found dup hits:	[('ENSG00000117262', 2), ('ENSG00000188092', 2), ('ENSG00000228044', 2), ('ENSG00000226506', 2), ('E
16 input query terms found no hit:	['ENSG00000177133', 'ENSG00000215695', 'ENSG00000272529', 'ENSG00000250410', 'ENSG00000280987', 'ENS


In [5]:
adata = adata[:, ~adata.var_names.str.startswith('ENSG')].copy()

print(f"Remaining genes after filtering out unmapped IDs: {adata.n_vars}")

Remaining genes after filtering out unmapped IDs: 27070


In [6]:
sc.pp.filter_genes(adata, min_cells=5)

In [8]:
ethnicity = adata.obs["self_reported_ethnicity"].astype("string").str.strip()
unknown_ethnicity = ethnicity.str.casefold().eq("unknown")
ethnicity = ethnicity.mask(unknown_ethnicity, "White")
adata.obs["self_reported_ethnicity"] = ethnicity.astype("category")

In [ ]:
adata = adata[adata.obs["self_reported_ethnicity"].notna()].copy()
adata.obs["self_reported_ethnicity"] = (
    adata.obs["self_reported_ethnicity"].cat.remove_unused_categories()
)

donor_counts = (
    adata.obs[["donor_id", "self_reported_ethnicity"]]
    .drop_duplicates()
    .groupby("self_reported_ethnicity", observed=True)["donor_id"]
    .nunique()
    .sort_values(ascending=False)
)
display(donor_counts.rename("n_donors").to_frame())
sc.pl.umap(adata, color="self_reported_ethnicity", use_raw=False)

In [ ]:
from src.scrna.pseudobulk_de import analyze_pseudobulk_ethnicity_de

CELL_TYPE = "cd4_naive"
MIN_DONORS_PER_GROUP = 3
MIN_CELLS_PER_PSEUDOBULK = 20

pseudobulk_de_analysis = analyze_pseudobulk_ethnicity_de(
    adata,
    cell_type=CELL_TYPE,
    min_donors_per_group=MIN_DONORS_PER_GROUP,
    min_cells_per_pseudobulk=MIN_CELLS_PER_PSEUDOBULK,
    alpha=0.05,
)
ethnicity_de_results = pseudobulk_de_analysis["results"]
significant_ethnicity_de = pseudobulk_de_analysis["significant"]
tissue_status_rows = pseudobulk_de_analysis["status"]

if significant_ethnicity_de.empty:
    print("No significant differential-expression results at global FDR q < 0.05.")
else:
    display(significant_ethnicity_de[[
        "tissue", "cell_type", "focal_ethnicity", "reference_ethnicity",
        "gene", "log2FoldChange", "pvalue", "padj",
        "padj_across_tissues", "n_focal_donors", "n_reference_donors",
    ]])

display(tissue_status_rows)


### Donor-level cell-type abundance by tissue and ethnicity

This summarizes cell counts and proportions for each donor in each tissue. The scCODA models compare compositions within tissue, use White as the reference ethnicity, adjust for sex when both sex groups are present, and report credible effects at 5% FDR within each tissue model. Results across tissues are not jointly FDR-adjusted. Groups with fewer than three donors in a tissue are omitted.


In [ ]:
MIN_CELLS_PER_SAMPLE = 100
MIN_DONORS_PER_GROUP = 3

abundance_analysis = analyze_donor_cell_type_abundance(
    adata,
    min_cells_per_sample=MIN_CELLS_PER_SAMPLE,
    min_donors_per_group=MIN_DONORS_PER_GROUP,
    alpha=0.05,
)
cell_type_counts = abundance_analysis["cell_type_counts"]
sample_metadata = abundance_analysis["sample_metadata"]
proportions_long = abundance_analysis["proportions_long"]
cell_type_abundance_summary = abundance_analysis["summary"]
sccoda_results_by_tissue = abundance_analysis["results_by_tissue"]
credible_cell_type_abundance = abundance_analysis["credible_findings"]
abundance_model_status = abundance_analysis["status"]

display(cell_type_abundance_summary)
if credible_cell_type_abundance.empty:
    print("No cell types had a credible ethnicity effect at scCODA FDR 0.05.")
else:
    display(credible_cell_type_abundance)
display(abundance_model_status)


In [ ]:
credible_effect_size_table = plot_credible_effect_sizes(
    sccoda_results_by_tissue, credible_cell_type_abundance
)
display(credible_effect_size_table)


### Donor-level boxplots for credible abundance findings

Each panel compares donor-level cell-type proportions for White and Hispanic or Latin donors in one tissue. Red brackets mark scCODA-credible effects; brown brackets mark Wilcoxon rank-sum comparisons that remain significant after Benjamini–Hochberg correction across all tested tissue–cell-type pairs. Points represent individual donors.


In [ ]:
plot_credible_abundance_findings(
    proportions_long, credible_cell_type_abundance
)


### CMV sensitivity analysis for scCODA findings

This follow-up keeps the original donor-tissue samples and cell-type reference where possible. It visualizes each original scCODA finding across ethnicity-by-CMV strata, then refits tissue-specific scCODA models with ethnicity, CMV, and sex (when variable). Sparse strata are reported explicitly; adjusted estimates are exploratory where ethnicity and CMV overlap poorly.


In [ ]:
sample_metadata_cmv, proportions_long_cmv = prepare_cmv_data(
    adata, sample_metadata, proportions_long
)

original_hits = credible_cell_type_abundance.drop_duplicates(
    ["tissue", "cell_type"]
).copy()
hit_tissues = original_hits["tissue"].drop_duplicates().tolist()
cmv_donor_counts = (
    sample_metadata_cmv.reset_index()
    .loc[lambda data: data["tissue"].isin(hit_tissues)
         & data["self_reported_ethnicity"].isin(["White", "Hispanic or Latin"])]
    .groupby(["tissue", "self_reported_ethnicity", "cmv"], observed=True)
    .agg(n_donors=("donor_id", "nunique"))
    .reset_index()
)
display(cmv_donor_counts)

plot_cmv_stratified_proportions(proportions_long_cmv, original_hits)

(
    cmv_adjusted_results_by_tissue,
    cmv_sensitivity_summary,
    cmv_adjusted_status,
) = fit_cmv_adjusted_models(
    sample_metadata_cmv,
    cell_type_counts,
    sccoda_results_by_tissue,
    original_hits,
    min_donors_per_group=MIN_DONORS_PER_GROUP,
    alpha=0.05,
)
display(cmv_sensitivity_summary)
display(cmv_adjusted_status)


### Compare unadjusted and CMV-adjusted scCODA findings

These panels reuse the CMV-stratified donor proportions above. Red brackets mark the original ethnicity effect; purple brackets mark effects that remain credible after CMV adjustment, and dashed gray brackets mark effects that do not remain credible.


In [ ]:
plot_cmv_sensitivity(cmv_sensitivity_summary, proportions_long_cmv)
display(cmv_sensitivity_summary[[
    "tissue", "cell_type", "original_log2_fold_change",
    "cmv_adjusted_log2_fold_change", "cmv_adjusted_credible",
    "cmv_effect_credible", "cmv_log2_fold_change",
]])
